# 04 - Advanced: Mecking-Kocks closure, energy balance, Schmid stratification

Three matrix-twin asymmetry analyses that compose multiple modules:

1. **Energy-balance closure**: matrix-twin elastic gap vs predicted twin-boundary cost.
2. **Mecking-Kocks variant k2 ratio**: from per-variant saturated dislocation densities.
3. **Schmid stratification**: do high-Schmid pairs show a larger twin-shear projection?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from midas_defect.energy import energy_balance_closure, twin_boundary_energy_density
from midas_defect.thermodynamics import variant_specific_k2, mk_evolve, taylor_implied_total_rho, wh_visible_fraction

## 1. Energy-balance closure

If the matrix carries more elastic energy than the twin and the gap is paid for by the
twin-boundary energy, the closure ratio should sit near unity. Deviations flag missing
energy reservoirs (dislocation cores, stacking faults, residual phase fraction) or a
constitutive-law mismatch.

In [ ]:
U_matrix, U_twin = 1.8e6, 1.0e6     # J/m^3   (planted gap)
gamma_Cu_TB = 0.04                  # J/m^2   (Cu twin boundary, Murr 1975)
L_lamella = 1.0e-7                  # m       (~100 nm)
closure = energy_balance_closure(U_matrix, U_twin, gamma_Cu_TB, L_lamella)
for k, v in closure.items(): print(f'{k:20s} {v:.3e}')

## 2. Mecking-Kocks variant k2 ratio

With per-variant saturated densities ``rho_sat = (k_1 / k_2)^2`` and the conservative
assumption that ``k_1`` is shared across variants (slip-system family is shared), the
ratio ``k_2_i / k_2_j`` lower-bounds the recovery-rate asymmetry.

In [ ]:
out = variant_specific_k2({'matrix': 7.7e12, 'twin': 4.0e12}, k1_literature=1.0e9)
print('k_ratio_per_variant:', out['k_ratio_per_variant'])
print('k2 absolute        :', out['k2_absolute'])
print('k2 ratios          :', out['k2_ratio_pairs'])
print('caveat             :', out['lower_bound_note'][:100], '...')
# Forward integrate MK to verify saturation:
eps_traj = np.linspace(0, 1.0, 200)
for variant in ('matrix', 'twin'):
    rho_t = mk_evolve(eps_traj, k1=1.0e9, k2=1.0e9 / out['k_ratio_per_variant'][variant], rho_init=1e10)
    plt.plot(eps_traj, rho_t, label=variant)
plt.xlabel('strain'); plt.ylabel('rho (m$^{-2}$)'); plt.yscale('log'); plt.legend(); plt.show()

## 3. Schmid-stratified twin-shear projection (mechanism signal)

Sort matched pairs into Schmid terciles, then plot the median twin-shear projection
per tercile. A monotone trend pinning the signal to active slip is the strongest
mechanism test.

In [ ]:
from midas_defect.schmid import stratify_pairs_by_schmid_max
rng = np.random.default_rng(0)
n_pairs = 120
schmid_per_grain = rng.uniform(0.2, 0.5, size=200)
pairs = rng.integers(0, 200, size=(n_pairs, 2))
dEps = (np.maximum(schmid_per_grain[pairs[:, 0]], schmid_per_grain[pairs[:, 1]]) - 0.30) * 0.02 \
       + rng.normal(scale=0.0015, size=n_pairs)
out = stratify_pairs_by_schmid_max(pairs, schmid_per_grain)
for t, idx in enumerate(out['tier_pair_indices']):
    print(f'T{t+1}: n={idx.size:3d}  schmid_range=[{out["tier_edges"][t]:.3f}, {out["tier_edges"][t+1]:.3f}]  '
          f'median dEps={np.median(dEps[idx]):+.4f}')

## 4. Bonus: Taylor visible-fraction sanity

If the modified-WH analysis recovers only a small fraction of the Taylor-inferred total
density, there is a sub-resolved fraction the LP probe is blind to.

In [ ]:
rho_total = taylor_implied_total_rho(7.0e8)
f_visible = wh_visible_fraction(7.7e12, 7.0e8)
print(f'rho Taylor at 700 MPa flow: {rho_total:.2e} m^-2')
print(f'WH-visible fraction       : {f_visible*100:.2f} %')